# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [2]:
%pip install ipywidgets


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
import numpy as np
import itertools

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [4]:
sup_df = pd.read_csv("../data/dayofweek.csv")

In [5]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df["dayofweek"] = sup_df["dayofweek"]
df

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3


In [6]:
X = df.drop('dayofweek', axis=1)
y = df["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [7]:
grid_param = {"kernel": ['linear', 'rbf', 'sigmoid'],
              'C': [0.01, 0.1, 1, 1.5, 5, 10],
              'gamma': ['scale', 'auto'],
              'class_weight': ['balanced', None]}

model = SVC(random_state=21, probability=True)
grid_search = GridSearchCV(estimator=model, param_grid=grid_param, n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
result = grid_search.cv_results_
result_svm = pd.DataFrame(result).sort_values('rank_test_score')

result_svm

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
70,0.659319,0.017558,0.015798,0.002162,10,None,auto,rbf,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.900000,0.848148,0.885185,0.884758,0.862454,0.876109,0.018419,1
64,0.813230,0.087296,0.020384,0.002975,10,balanced,auto,rbf,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.877778,0.851852,0.862963,0.873606,0.851301,0.863500,0.010870,2
58,0.661320,0.031018,0.023847,0.005651,5,None,auto,rbf,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.825926,0.811111,0.818519,0.821561,0.802974,0.816018,0.008116,3
52,0.701413,0.058888,0.024743,0.007143,5,balanced,auto,rbf,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.844444,0.785185,0.792593,0.817844,0.802974,0.808608,0.021007,4
63,59.288254,5.099102,0.015092,0.002610,10,balanced,auto,linear,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.729630,0.700000,0.755556,0.754647,0.665428,0.721052,0.034438,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53,0.923741,0.052384,0.028736,0.004835,5,balanced,auto,sigmoid,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.144444,0.148148,0.137037,0.126394,0.092937,0.129792,0.019869,68
65,0.809919,0.036265,0.027607,0.004384,10,balanced,auto,sigmoid,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.122222,0.140741,0.129630,0.100372,0.085502,0.115693,0.020052,69
41,0.982137,0.028130,0.027357,0.002027,1.5,balanced,auto,sigmoid,"{'C': 1.5, 'class_weight': 'balanced', 'gamma'...",0.066667,0.085185,0.081481,0.078067,0.085502,0.079380,0.006913,70
17,1.081851,0.072342,0.030924,0.002588,0.1,balanced,auto,sigmoid,"{'C': 0.1, 'class_weight': 'balanced', 'gamma'...",0.062963,0.066667,0.062963,0.059480,0.059480,0.062310,0.002678,71


In [ ]:
param_svm = grid_search.best_params_
param_svm

{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
param_grid = {'max_depth': [i for i in range(1,49)],
              'class_weight': ['balanced', None],
              'criterion': ['entropy', 'gini']}

model = DecisionTreeClassifier(random_state=21)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...]})

In [ ]:
result = grid_search.cv_results_
result_tree = pd.DataFrame(result).sort_values('rank_test_score')

result_tree

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
68,0.007622,0.003794,0.001865,0.000763,balanced,gini,21,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.859259,0.903704,0.884758,0.832714,0.873865,0.025066,1
72,0.005456,0.001597,0.001216,0.000110,balanced,gini,25,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.874074,0.903704,0.873606,0.828996,0.873854,0.025018,2
69,0.008610,0.002879,0.002125,0.001196,balanced,gini,22,"{'class_weight': 'balanced', 'criterion': 'gin...",0.885185,0.862963,0.903704,0.881041,0.828996,0.872378,0.025263,3
95,0.009056,0.003260,0.002506,0.002359,balanced,gini,48,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.828996,0.872372,0.025179,4
93,0.006839,0.002208,0.002187,0.001128,balanced,gini,46,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.828996,0.872372,0.025179,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50,0.005291,0.001656,0.003640,0.002781,balanced,gini,3,"{'class_weight': 'balanced', 'criterion': 'gin...",0.388889,0.303704,0.403704,0.427509,0.345725,0.373906,0.044064,188
144,0.003876,0.002238,0.001699,0.001002,None,gini,1,"{'class_weight': None, 'criterion': 'gini', 'm...",0.370370,0.351852,0.359259,0.353160,0.342007,0.355330,0.009338,189
96,0.001912,0.000172,0.001037,0.000088,None,entropy,1,"{'class_weight': None, 'criterion': 'entropy',...",0.370370,0.351852,0.359259,0.353160,0.342007,0.355330,0.009338,189
48,0.004401,0.001038,0.001430,0.000197,balanced,gini,1,"{'class_weight': 'balanced', 'criterion': 'gin...",0.262963,0.318519,0.266667,0.323420,0.260223,0.286358,0.028376,191


In [ ]:
param_tree = grid_search.best_params_
param_tree

{'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21}

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [ ]:
param_grid = {'n_estimators': [5, 10, 50, 100],
              'max_depth': [i for i in range(1,50)],
              'class_weight': ['balanced', None],
              'criterion': ['entropy', 'gini']}

model = RandomForestClassifier(random_state=21)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                       13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
                                       23, 24, 25, 26, 27, 28, 29, 30, ...],
                         'n_estimators': [5, 10, 50, 100]})

In [ ]:
result = grid_search.cv_results_
result_forest = pd.DataFrame(result).sort_values('rank_test_score')

result_forest

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
95,0.219964,0.016849,0.007477,0.000360,balanced,entropy,24,100,"{'class_weight': 'balanced', 'criterion': 'ent...",0.922222,0.900000,0.903704,0.910781,0.884758,0.904293,0.012361,1
698,0.119716,0.007937,0.006833,0.001619,None,gini,28,50,"{'class_weight': None, 'criterion': 'gini', 'm...",0.922222,0.900000,0.907407,0.903346,0.888476,0.904290,0.010961,2
314,0.109135,0.007692,0.007098,0.002333,balanced,gini,30,50,"{'class_weight': 'balanced', 'criterion': 'gin...",0.922222,0.903704,0.900000,0.907063,0.884758,0.903549,0.012056,3
711,0.235339,0.019147,0.009761,0.002394,None,gini,31,100,"{'class_weight': None, 'criterion': 'gini', 'm...",0.918519,0.911111,0.900000,0.910781,0.877323,0.903547,0.014380,4
115,0.244591,0.014303,0.008261,0.001356,balanced,entropy,29,100,"{'class_weight': 'balanced', 'criterion': 'ent...",0.922222,0.900000,0.907407,0.907063,0.881041,0.903547,0.013380,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392,0.011199,0.001065,0.002857,0.001543,None,entropy,1,5,"{'class_weight': None, 'criterion': 'entropy',...",0.355556,0.366667,0.374074,0.345725,0.327138,0.353832,0.016467,780
4,0.012151,0.002265,0.002327,0.001488,balanced,entropy,2,5,"{'class_weight': 'balanced', 'criterion': 'ent...",0.318519,0.366667,0.381481,0.353160,0.345725,0.353110,0.021165,781
200,0.009317,0.003496,0.001378,0.000055,balanced,gini,2,5,"{'class_weight': 'balanced', 'criterion': 'gin...",0.311111,0.377778,0.377778,0.353160,0.312268,0.346419,0.029749,782
196,0.010773,0.004458,0.001487,0.000120,balanced,gini,1,5,"{'class_weight': 'balanced', 'criterion': 'gin...",0.262963,0.292593,0.285185,0.282528,0.293680,0.283390,0.011062,783


In [ ]:
param_forest = grid_search.best_params_
param_forest

{'class_weight': 'balanced',
 'criterion': 'entropy',
 'max_depth': 24,
 'n_estimators': 100}

## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [8]:
n_estimators = [5, 10, 50, 100]
max_depth = [i for i in range(1,50)]
class_weight = ['balanced', None]
criterion = ['entropy', 'gini']

results = []
params = list(itertools.product(n_estimators, max_depth, class_weight, criterion))

for comb_params in tqdm(params):
    n_estimators, max_depth, class_weight, criterion = comb_params
    model = RandomForestClassifier(random_state=21, n_estimators = n_estimators, max_depth=max_depth, class_weight=class_weight, criterion=criterion)
    result = cross_val_score(model, X, y, cv=5, n_jobs=-1, verbose=0, scoring="accuracy")
    results.append({'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'class_weight': class_weight,
                    'criterion': criterion, 
                     'random_state': 21, 
                     'mean_accuracy': np.mean(result), 
                     'std_accuracy': np.std(result)})

results_for_forest = pd.DataFrame(results)
results_for_forest = results_for_forest.sort_values('mean_accuracy', ascending=False)

results_for_forest

,n_estimators,max_depth,class_weight,criterion,random_state,mean_accuracy,std_accuracy
642,100,14,None,entropy,21,0.562369,0.152139
446,50,14,None,entropy,21,0.551109,0.165483
654,100,17,None,entropy,21,0.546958,0.156132
452,50,16,balanced,entropy,21,0.545179,0.167796
660,100,19,balanced,entropy,21,0.541624,0.168721
...,...,...,...,...,...,...,...
13,5,4,balanced,gini,21,0.243213,0.048634
5,5,2,balanced,gini,21,0.224222,0.048249
0,5,1,balanced,entropy,21,0.198709,0.045986
1,5,1,balanced,gini,21,0.193962,0.047020


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [ ]:
best_model = RandomForestClassifier(random_state=21, class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100)
best_model.fit(X_train, y_train)

accuracy = best_model.score(X_test, y_test)
accuracy

0.9260355029585798